# Kaggle Submission — XGBoost (best model, lowest holdout WMAE)

Generates a Kaggle-format submission using the already-trained, saved
XGBoost pipeline (`models/xgboost_pipeline.joblib`) from
`model_experiment_XGBoost.ipynb` — no retraining needed here, this just
loads the pipeline and runs it on the real Kaggle `test.csv`.

**Why XGBoost**: of every model tried in this project, it has the lowest
local-test holdout WMAE:

| Model | Holdout WMAE |
|---|---|
| **XGBoost** | **1639.12** |
| LightGBM | 1672.26 |
| N-BEATS (best generic) | 2161.25 |
| PatchTST | 2190.61 |
| DLinear | 2532.49 |
| TimesFM (direct) | 2618.40 |
| TimesFM (recursive) | 2787.23 |
| ARIMA | 2579.07 (different eval window, not directly comparable) |
| TFT (reduced-scope, no tuning) | 3996.37 |
| Prophet | 6932.91 |

The saved pipeline is a plain `sklearn.pipeline.Pipeline`
(`FeatureEngineeringTransformer` -> `FeatureSelector` -> `XGBRegressor`),
fit on all of `train.csv` at the tuned hyperparameters — it takes bare
`Store/Dept/Date/IsHoliday` rows straight from `test.csv`, no
pre-computation needed by the caller.

<a id='1'></a>
## 1. Load pipeline and test data

In [1]:
import joblib
import numpy as np
import pandas as pd

DATA_DIR = 'data/raw/walmart-recruiting-store-sales-forecasting/'

pipeline = joblib.load('models/xgboost_pipeline.joblib')
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=['Date'])
sample_submission = pd.read_csv(DATA_DIR + 'sampleSubmission.csv')

print(f'test.csv: {test.shape}')
print(f'sampleSubmission.csv: {sample_submission.shape}')
print(test.head())

test.csv: (115064, 4)
sampleSubmission.csv: (115064, 2)
   Store  Dept       Date  IsHoliday
0      1     1 2012-11-02      False
1      1     1 2012-11-09      False
2      1     1 2012-11-16      False
3      1     1 2012-11-23       True
4      1     1 2012-11-30      False


<a id='2'></a>
## 2. Predict

Raw `test.csv` rows straight in, no pre-computed features -- the
pipeline's own `FeatureEngineeringTransformer` handles that internally,
same as it did for `local_test_raw` during the notebook's own holdout
evaluation. Predictions clipped to >=0, matching the convention used
everywhere else in this project (`Weekly_Sales` forecasts shouldn't go
negative even though ~0.3% of actual historical values are, from real
merchandise returns).

In [2]:
preds = pipeline.predict(test)
preds = np.clip(preds, 0, None)

print(f'{len(preds)} predictions generated')
print(f'min={preds.min():.2f}, max={preds.max():.2f}, mean={preds.mean():.2f}')
print(f'any NaN: {np.isnan(preds).any()}')

115064 predictions generated
min=0.00, max=184099.73, mean=11518.61
any NaN: False


<a id='3'></a>
## 3. Format submission

Kaggle's required format: `Id = f'{Store}_{Dept}_{Date}'`,
`Weekly_Sales = <predicted value>` -- verified directly against
`sampleSubmission.csv`'s own `Id` format below before writing anything.

In [3]:
submission = pd.DataFrame({
    'Id': test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': preds,
})

# sanity checks against the official sample submission before trusting this
assert len(submission) == len(sample_submission), \
    f'row count mismatch: {len(submission)} vs {len(sample_submission)}'
assert set(submission['Id']) == set(sample_submission['Id']), \
    'Id values do not match sampleSubmission.csv exactly'
assert submission['Weekly_Sales'].isna().sum() == 0, 'NaN predictions present'
assert (submission['Weekly_Sales'] >= 0).all(), 'negative predictions present'

print('All sanity checks passed: row count, Id format, no NaN, no negatives.')
submission.head()

All sanity checks passed: row count, Id format, no NaN, no negatives.


,Id,Weekly_Sales
0,1_1_2012-11-02,36453.011719
1,1_1_2012-11-09,20558.042969
2,1_1_2012-11-16,20637.978516
3,1_1_2012-11-23,19130.212891
4,1_1_2012-11-30,21333.226562


<a id='4'></a>
## 4. Save

In [4]:
submission.to_csv('submission.csv', index=False)
print('Saved to submission.csv -- ready to upload to Kaggle.')
print(f'{len(submission)} rows')

Saved to submission.csv -- ready to upload to Kaggle.
115064 rows
